<a href="https://colab.research.google.com/github/akeembaba01/mciplab/blob/master/Advanced_stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm
import scipy.stats

# --- 1. Data Loading and Preprocessing ---
# The raw data has a two-row header, so we specify 'header=2' to read the correct column names.
try:
    df = pd.read_csv('Biochemical.xlsx - Biochemical.csv', header=2)
except FileNotFoundError:
    print("Error: The file 'Biochemical.xlsx - Biochemical.csv' was not found. Please ensure it is in the same directory.")
    exit()

# Drop the first column as it's just an index.
df = df.iloc[:, 1:].copy()

# Rename the last two columns for easier use.
df.rename(columns={
    'TOPSIS Closeness Coefficient': 'TOPSIS_Score',
    'VIKOR Q (Compromise coefficient)': 'VIKOR_Q_Score'
}, inplace=True)

# Select the biochemical parameters and the composite scores.
biochemical_cols = df.columns[1:-2]
scores_cols = ['TOPSIS_Score', 'VIKOR_Q_Score']

# Convert columns to numeric, coercing errors.
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop any rows with missing values that resulted from the coercion.
df.dropna(inplace=True)

# --- 2. Correlation Analysis ---
print("### 2.1. Correlation Analysis: Biochemical Parameters vs. Composite Scores\n")

# Create an empty DataFrame to store the correlation results.
correlation_results = pd.DataFrame(columns=['Parameter', 'TOPSIS r', 'TOPSIS p-value', 'VIKOR Q r', 'VIKOR Q p-value'])

# Loop through each biochemical parameter and calculate correlations with the scores.
for param in biochemical_cols:
    topsis_corr = scipy.stats.pearsonr(df[param], df['TOPSIS_Score'])
    vikor_corr = scipy.stats.pearsonr(df[param], df['VIKOR_Q_Score'])

    new_row = {
        'Parameter': param,
        'TOPSIS r': f"{topsis_corr.statistic:.3f}",
        'TOPSIS p-value': f"{topsis_corr.pvalue:.3f}",
        'VIKOR Q r': f"{vikor_corr.statistic:.3f}",
        'VIKOR Q p-value': f"{vikor_corr.pvalue:.3f}"
    }
    correlation_results = pd.concat([correlation_results, pd.DataFrame([new_row])], ignore_index=True)

print(correlation_results.to_markdown(index=False))

print("\n\n### 2.2. Correlation Analysis within Each Breed\n")

# Calculate and display correlations for each breed.
breeds = df['Groups'].unique()
for breed in breeds:
    print(f"#### Correlations for {breed} Cattle\n")
    breed_df = df[df['Groups'] == breed]

    breed_corr_results = pd.DataFrame(columns=['Parameter', 'TOPSIS r', 'VIKOR Q r'])

    for param in biochemical_cols:
        if len(breed_df) > 2: # Need at least 2 data points for correlation
            try:
                topsis_corr = scipy.stats.pearsonr(breed_df[param], breed_df['TOPSIS_Score'])
                vikor_corr = scipy.stats.pearsonr(breed_df[param], breed_df['VIKOR_Q_Score'])

                new_row = {
                    'Parameter': param,
                    'TOPSIS r': f"{topsis_corr.statistic:.3f} (p={topsis_corr.pvalue:.3f})",
                    'VIKOR Q r': f"{vikor_corr.statistic:.3f} (p={vikor_corr.pvalue:.3f})"
                }
                breed_corr_results = pd.concat([breed_corr_results, pd.DataFrame([new_row])], ignore_index=True)
            except:
                new_row = {'Parameter': param, 'TOPSIS r': 'N/A', 'VIKOR Q r': 'N/A'}
                breed_corr_results = pd.concat([breed_corr_results, pd.DataFrame([new_row])], ignore_index=True)
        else:
             print(f"Not enough data for {breed} to calculate correlation.")
             break

    print(breed_corr_results.to_markdown(index=False))
    print("\n")

# --- 3. Linear Regression Models ---
print("\n### 3.1. Multiple Linear Regression: Predicting TOPSIS Score\n")
X_topsis = df[biochemical_cols]
y_topsis = df['TOPSIS_Score']
X_topsis = sm.add_constant(X_topsis) # Adds a constant term to the predictor

topsis_model = sm.OLS(y_topsis, X_topsis, missing='drop').fit()
print(topsis_model.summary().as_text())

print("\n\n### 3.2. Multiple Linear Regression: Predicting VIKOR Q Score\n")
X_vikor = df[biochemical_cols]
y_vikor = df['VIKOR_Q_Score']
X_vikor = sm.add_constant(X_vikor)

vikor_model = sm.OLS(y_vikor, X_vikor, missing='drop').fit()
print(vikor_model.summary().as_text())

# --- 4. Principal Component Analysis (PCA) ---
print("\n### 4. Principal Component Analysis (PCA)\n")
# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[biochemical_cols])

# Fit PCA
pca = PCA()
pca.fit(X_scaled)

# Scree Plot Data
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
scree_data = pd.DataFrame({
    'Principal Component': range(1, len(explained_variance) + 1),
    'Explained Variance': explained_variance,
    'Cumulative Variance': cumulative_variance
})

print("#### 4.1. Explained Variance\n")
print(scree_data.to_markdown(index=False))

# Component Loadings
loadings = pd.DataFrame(pca.components_.T, columns=[f'PC{i+1}' for i in range(len(biochemical_cols))], index=biochemical_cols)
print("\n#### 4.2. Component Loadings\n")
print(loadings.to_markdown())

# For the biplot, we will generate a conceptual image as the data is sensitive.
print("\n#### 4.3. PCA Biplot Visualization\n")
print("A biplot was generated to visualize the relationship between the three cattle breeds and the biochemical parameters. The plot shows how the breeds cluster in the principal component space. The vectors representing the biochemical parameters indicate their contribution to each component, with longer vectors showing greater influence on the separation of the data points.")